In [ ]:
from datasets import load_dataset

ds = load_dataset("K-Monitor/kmdb_base")

In [ ]:
ds['train'][-1]

In [ ]:
# --- Configuration -----------------------------------------------------------
TARGET_TAG = "kampány közpénzből"

# Paste news_id-s of known articles that carry the tag
POSITIVE_IDS = [
    # e.g. 65148,
    65024, 64982, 64862, 65035, 65120, 65155, 65188, 65180, 60099, 64654
]

# How many random negatives to sample per positive (class balance)
NEG_RATIO = 10

# Minimum classifier probability to surface an article as a candidate
THRESHOLD = 0.5

In [ ]:
import random
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

random.seed(42)
np.random.seed(42)

# Flatten the train split into a list of dicts for easy processing
articles = list(ds["train"])
print(f"Total articles: {len(articles)}")

# Show the tag distribution around the target tag
from collections import Counter
tag_counts = Counter(tag for a in articles for tag in a["others"])
print(f"\nTop 20 'others' tags:")
for tag, cnt in tag_counts.most_common(20):
    print(f"  {cnt:5d}  {tag}")
print(f"\nArticles already tagged '{TARGET_TAG}': {tag_counts.get(TARGET_TAG, 0)}")

In [ ]:
# --- Build positive set -------------------------------------------------------
# Positives = articles in POSITIVE_IDS + any already tagged with TARGET_TAG
id_to_article = {a["news_id"]: a for a in articles}

positives_from_ids = [id_to_article[i] for i in POSITIVE_IDS if i in id_to_article]
missing_ids = [i for i in POSITIVE_IDS if i not in id_to_article]
if missing_ids:
    print(f"WARNING: {len(missing_ids)} news_ids not found in dataset: {missing_ids}")

positives_already_tagged = [a for a in articles if TARGET_TAG in a["others"]]

# Merge, deduplicate by news_id
seen_ids = set()
positives = []
for a in positives_from_ids + positives_already_tagged:
    if a["news_id"] not in seen_ids:
        positives.append(a)
        seen_ids.add(a["news_id"])

pos_ids = {a["news_id"] for a in positives}
print(f"Positive examples: {len(positives)}")

# --- Build negative set -------------------------------------------------------
negatives_pool = [a for a in articles if a["news_id"] not in pos_ids]
n_neg = min(len(negatives_pool), len(positives) * NEG_RATIO)
negatives = random.sample(negatives_pool, n_neg)
print(f"Negative examples sampled: {len(negatives)}")

In [ ]:
# --- Compute embeddings -------------------------------------------------------
# paraphrase-multilingual-mpnet-base-v2 handles Hungarian well
MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"
model = SentenceTransformer(MODEL_NAME)

def article_text(a: dict) -> str:
    """Combine title + description into a single string to embed."""
    parts = [a.get("title") or "", a.get("description") or ""]
    return " ".join(p for p in parts if p).strip()

train_articles = positives + negatives
train_labels   = [1] * len(positives) + [0] * len(negatives)
train_texts    = [article_text(a) for a in train_articles]

print("Encoding training set …")
train_embeddings = model.encode(train_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
print(f"Embedding shape: {train_embeddings.shape}")

In [ ]:
# --- Train binary classifier --------------------------------------------------
clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0),
)
clf.fit(train_embeddings, train_labels)

# Quick cross-validation estimate (only meaningful when positives >= ~10)
if len(positives) >= 5:
    scores = cross_val_score(clf, train_embeddings, train_labels, cv=min(5, len(positives)), scoring="f1")
    print(f"CV F1: {scores.mean():.3f} ± {scores.std():.3f}")
else:
    print(f"Only {len(positives)} positives — skipping CV, results are indicative only.")

In [ ]:
# --- Predict on ALL articles --------------------------------------------------
all_texts = [article_text(a) for a in articles]
print(f"Encoding {len(all_texts)} articles …")
all_embeddings = model.encode(all_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

probs = clf.predict_proba(all_embeddings)[:, 1]   # P(positive)

df = pd.DataFrame({
    "news_id":    [a["news_id"]    for a in articles],
    "title":      [a["title"]      for a in articles],
    "source_url": [a["source_url"] for a in articles],
    "pub_time":   [a["pub_time"]   for a in articles],
    "others":     [a["others"]     for a in articles],
    "prob":       probs,
    "is_known_positive": [a["news_id"] in pos_ids for a in articles],
})
print(df["prob"].describe())

In [ ]:
# --- Candidates: not yet tagged, classifier thinks positive -------------------
candidates = (
    df[~df["is_known_positive"] & (df["prob"] >= THRESHOLD)]
    .sort_values("prob", ascending=False)
    .reset_index(drop=True)
)
print(f"Candidate articles (prob ≥ {THRESHOLD}): {len(candidates)}\n")
pd.set_option("display.max_colwidth", 80)
candidates[["prob", "news_id", "pub_time", "title", "source_url"]].head(30)

In [ ]:
# --- Outliers among KNOWN positives (low classifier score) --------------------
# These are articles you provided as positive examples but the model found 
# them dissimilar from the rest — worth reviewing (possible label errors or
# edge cases that broaden the concept).
outlier_positives = (
    df[df["is_known_positive"] & (df["prob"] < THRESHOLD)]
    .sort_values("prob")
    .reset_index(drop=True)
)
print(f"Known-positive outliers (prob < {THRESHOLD}): {len(outlier_positives)}\n")
outlier_positives[["prob", "news_id", "pub_time", "title", "source_url"]]

In [ ]:
# --- Export candidates + scores for human review ------------------------------
out_path = "kampany_kozpenzbol_candidates.csv"
candidates.to_csv(out_path, index=False)
print(f"Saved {len(candidates)} candidates → {out_path}")